# CMB physics: from expansion to recombination and acoustic oscillations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daanmeerburg/RTG_3186_CMB_Lectures/blob/main/CMB_physics_from_expansion_to_acoustics.ipynb)

This notebook accompanies the revised **Lecture 1** and assumes no previous numerical cosmology. Run the cells in order. We will:

1. distinguish local energy conservation from constant energy in a comoving volume;
2. define comoving distance and conformal time before using them;
3. derive and solve the Saha equation, including the temperature in eV;
4. construct an approximate visibility function and calculate the sound horizon;
5. explain scalar metric potentials and anisotropic stress;
6. demonstrate momentum exchange and velocity locking;
7. derive the temperature oscillator, its effective temperature, and its shifted equilibrium;
8. connect coherent acoustic phases to the angular pattern.

The Saha calculation describes equilibrium hydrogen recombination. It does not replace a non-equilibrium recombination calculation. The constant-potential acoustic model isolates the physics; it is not a full CMB spectrum calculation.

## 1. Imports, constants, and conventions

The code uses NumPy, SciPy, and Matplotlib, available in Google Colab. The lecture uses natural units; the background calculations here restore $c$, $k_B$, and $\hbar$ and use SI units, with distances reported in Mpc.

$T_0$ is today's mean temperature. $\Theta_0=\delta_\gamma/4$ is a temperature perturbation; lowercase $\theta_i=i\mathbf{k}\cdot\mathbf{v}_i$ is a velocity divergence. Dots in the perturbation equations mean conformal-time derivatives. The final oscillator examples use explicitly defined dimensionless time variables.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, quad, solve_ivp
from scipy.optimize import brentq
from scipy.special import zeta

plt.rcParams.update({
    'figure.figsize': (9.5, 5.5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

# Physical constants (SI)
c = 299_792_458.0                 # m/s
k_B = 1.380649e-23                # J/K
hbar = 1.054571817e-34            # J s
G = 6.67430e-11                   # m^3 kg^-1 s^-2
m_e = 9.1093837015e-31            # kg
m_p = 1.67262192369e-27           # kg
sigma_T = 6.6524587321e-29        # m^2
eV = 1.602176634e-19              # J
Mpc = 3.085677581491367e22        # m

# A representative flat Lambda-CDM background (close to Planck 2018).
h = 0.674
H0 = 100.0 * h * 1000.0 / Mpc    # s^-1
Omega_b = 0.0493
Omega_c = 0.264
Omega_m = Omega_b + Omega_c
Omega_gamma_h2 = 2.4728e-5        # photons for T0 = 2.7255 K
Omega_gamma = Omega_gamma_h2 / h**2
N_eff = 3.046
Omega_r = Omega_gamma * (1.0 + 0.2271 * N_eff)
Omega_Lambda = 1.0 - Omega_m - Omega_r
T0 = 2.7255                       # K
Y_p = 0.245                       # helium mass fraction

rho_crit0 = 3.0 * H0**2 / (8.0 * np.pi * G)
n_H0 = (1.0 - Y_p) * Omega_b * rho_crit0 / m_p

print(f'H0 = {H0 * Mpc / 1000:.2f} km/s/Mpc')
print(f'Omega_r = {Omega_r:.3e}, Omega_m = {Omega_m:.3f}, Omega_Lambda = {Omega_Lambda:.3f}')
print(f'Present hydrogen-nucleus density = {n_H0:.3f} m^-3')

def H_of_z(z):
    return H0 * np.sqrt(Omega_r*(1+z)**4 + Omega_m*(1+z)**3 + Omega_Lambda)


## 2. Expansion history (lecture Sections 2–3)

For a flat Universe,

$$H^2(a)=H_0^2\left(\Omega_r a^{-4}+\Omega_m a^{-3}+\Omega_\Lambda\right).$$

The following plot shows each contribution to $H^2/H_0^2$. The $\Omega_i$ are present-day density fractions. Equality occurs where two curves cross.

In [ ]:
a = np.logspace(-7, 0, 1200)
radiation_term = Omega_r * a**-4
matter_term = Omega_m * a**-3
vacuum_term = Omega_Lambda * np.ones_like(a)

a_eq = Omega_r / Omega_m
z_eq = 1.0 / a_eq - 1.0
a_Lambda = (Omega_m / Omega_Lambda)**(1.0 / 3.0)

fig, ax = plt.subplots()
ax.loglog(a, radiation_term, label=r'radiation $\Omega_r a^{-4}$')
ax.loglog(a, matter_term, label=r'matter $\Omega_m a^{-3}$')
ax.loglog(a, vacuum_term, label=r'vacuum $\Omega_\Lambda$')
ax.axvline(a_eq, color='0.3', ls='--', label=fr'matter–radiation equality: $z\simeq{z_eq:.0f}$')
ax.axvline(a_Lambda, color='0.5', ls=':', label='matter–vacuum equality')
ax.set(xlabel='Scale factor $a$', ylabel=r'Contribution to $H^2/H_0^2$',
       title='The components dominating the expansion')
ax.legend(fontsize=10)
plt.show()

**Check your understanding.** Why does radiation dominate at sufficiently small $a$ even though $\Omega_r$ is tiny today? What physical effect supplies the extra factor of $a^{-1}$ relative to matter?

### 2.1 Local conservation and energy in a comoving volume

The local law $\nabla_\mu T^{\mu\nu}=0$ gives $d\rho/dt+3H(\rho+P)=0$. For a region of fixed comoving volume, $V\propto a^3$ and $E=\rho V$, so $dE=-P\,dV$.

A general expanding FLRW spacetime does not have the time-translation symmetry needed for a globally conserved energy of the usual kind. Local covariant conservation therefore does not imply constant energy in this region. Matter has $E_m\propto a^0$, radiation $E_r\propto a^{-1}$, and vacuum $E_\Lambda\propto a^3$.

We normalize each component to its own energy at $a=1$ below. The curves compare time dependence, not the relative abundance of components.

In [ ]:
a_energy = np.logspace(-3, 0, 400)
fig, ax = plt.subplots()
for w, label in [(0, 'matter'), (1/3, 'radiation'), (-1, 'vacuum')]:
    # rho ~ a^[-3(1+w)], V ~ a^3, hence E ~ a^(-3w).
    ax.loglog(a_energy, a_energy**(-3*w), label=label)
ax.set(xlabel='Scale factor $a$', ylabel=r'$E(a)/E(1)$ in a fixed comoving volume',
       title='Local conservation does not require constant comoving-volume energy')
ax.legend()
plt.show()


### 2.2 Define the distance before using it

Let $\chi=|\mathbf{x}|$ be the comoving radial coordinate, with us at $\chi=0$. The physical distance on a constant-time slice is $a\chi$. Define conformal time by $d\tau=dt/a$; a radial photon in the flat FLRW metric obeys $|d\chi|=c\,d\tau$. An incoming photon has $d\chi/d\tau=-c$.

Consequently the positive distance to its emission point is

$$\chi(z)=c[\tau_0-\tau(z)]=\int_0^z\frac{c\,dz'}{H(z')}.$$

A transverse comoving patch of size $L_{\rm com}$ subtends a small angle $L_{\rm com}/\chi$. The Hubble length $c/H$ is an expansion scale, while the particle horizon integrates the full past light-travel distance; they need not coincide. We introduce Fourier modes only when discussing perturbations below.

In [ ]:
z_distance_example = 1100.0
chi_example_Mpc = quad(lambda z: c/(Mpc*H_of_z(z)), 0, z_distance_example)[0]
print(f'Comoving distance to z=1100: {chi_example_Mpc:.1f} Mpc')
print(f'Physical radial distance on the emission-time slice: '
      f'{chi_example_Mpc/(1+z_distance_example):.2f} Mpc')
print(f'Conformal lookback c*(tau_0-tau) = {chi_example_Mpc:.1f} Mpc')
print('This distance example uses a chosen redshift, not a Saha prediction.')


## 3. Deriving and solving Saha equilibrium (lecture Section 5.2)

Let $n_H=n_p+n_{H^0}$ count **all hydrogen nuclei**, while $n_{H^0}$ counts neutral atoms. Neglecting free electrons from helium, charge neutrality gives $n_e=n_p$. Then $x_e=n_e/n_H$, $n_p=x_en_H$, and $n_{H^0}=(1-x_e)n_H$. We retain the helium mass fraction when calculating the number of hydrogen nuclei.

For $p+e^-\leftrightarrow H+\gamma$, chemical equilibrium requires $\mu_p+\mu_e=\mu_{H^0}$ because $\mu_\gamma=0$. A dilute, non-relativistic species has

$$n_i=g_i\left(\frac{m_i k_BT}{2\pi\hbar^2}\right)^{3/2}
\exp\left(\frac{\mu_i-m_i c^2}{k_BT}\right).$$

In $n_en_p/n_{H^0}$ the chemical potentials cancel. Using $(m_e+m_p-m_{H^0})c^2=E_I$ and $m_{H^0}\simeq m_p$ in the prefactor gives

$$\frac{x_e^2}{1-x_e}
=\frac{1}{n_H}\left(\frac{m_e k_BT}{2\pi\hbar^2}\right)^{3/2}
e^{-E_I/(k_BT)}\equiv y.$$

The ground-state spin factor is $g_eg_p/g_{H^0}=2\times2/4=1$. The stable positive solution is $x_e=2/[1+\sqrt{1+4/y}]$.

**Why not 13.6 eV?** That is the atomic binding energy, suggesting a naive threshold estimate; it is not the recombination temperature predicted by Saha. There are roughly a billion photons per baryon. With $\eta_H=n_H/n_\gamma$,

$$y=\frac{\pi^2}{2\zeta(3)\eta_H}
\left(\frac{m_ec^2}{2\pi k_BT}\right)^{3/2}e^{-E_I/(k_BT)}.$$

The large prefactor must be offset by a small exponential. We will solve for $x_e=1/2$ and compare this temperature with $E_I$, rather than equating the two.

In [ ]:
E_ion = 13.6 * eV

def saha_log_y(z):
    z = np.asarray(z, dtype=float)
    T = T0 * (1.0 + z)
    n_H = n_H0 * (1.0 + z)**3
    return (
        1.5 * np.log(m_e * k_B * T / (2.0 * np.pi * hbar**2))
        - np.log(n_H) - E_ion / (k_B * T)
    )

def saha_xe(z):
    # This expression avoids cancellation near x_e = 0.
    y = np.exp(np.clip(saha_log_y(z), -700, 700))
    return 2.0 / (1.0 + np.sqrt(1.0 + 4.0 / y))

z_rec = np.linspace(500, 2000, 2500)
xe_rec = saha_xe(z_rec)
z_half = brentq(lambda z: saha_log_y(z) - np.log(0.5), 500, 2000)
T_half_eV = k_B * T0 * (1 + z_half) / eV
n_gamma0 = 2 * zeta(3) / np.pi**2 * (k_B * T0 / (hbar * c))**3
eta_H = n_H0 / n_gamma0
z_binding = E_ion / (k_B * T0) - 1

fig, ax = plt.subplots()
ax.semilogy(z_rec, xe_rec, color='tab:blue', lw=2)
ax.axvline(z_half, color='tab:red', ls='--',
           label=fr'$x_e=0.5$ at $z\simeq{z_half:.0f}$')
ax.set(xlabel='Redshift $z$', ylabel=r'Free-electron fraction $x_e$',
       title='Hydrogen recombination in Saha equilibrium', ylim=(1e-5, 1.2))
ax.legend()
plt.show()
print(f'Photons per hydrogen nucleus = {1/eta_H:.3e}')
print(f'Half ionization: z = {z_half:.1f}, T = {T0*(1+z_half):.0f} K')
print(f'k_B T = {T_half_eV:.3f} eV; E_I/(k_B T) = {E_ion/(T_half_eV*eV):.2f}')
print(f'At k_B T = 13.6 eV: y = {np.exp(saha_log_y(z_binding)):.3e} (x_e nearly 1)')
assert abs(saha_xe(z_half) - 0.5) < 1e-8


**Check your understanding.** At $k_BT=13.6$ eV, is the gas mostly neutral or ionized? Use the printed value of $y$ to explain the answer. At half ionization, how large is $E_I/(k_BT)$? Explain how the photon-to-hydrogen ratio delays neutral-atom formation. Half ionization in equilibrium is not the same criterion as last scattering.

## 4. Optical depth and the visibility function

Restoring $c$ with conformal time measured in seconds,

$$\widetilde\tau(\tau)=\int_\tau^{\tau_0}a n_e\sigma_T c\,d\tau',
\qquad \dot{\widetilde\tau}=-a n_e\sigma_T c<0.$$

The tilde distinguishes **optical depth** from conformal time. In redshift variables,

$$\widetilde\tau(z)=\int_0^z\frac{c\,n_e(z')\sigma_T}{(1+z')H(z')}\,dz'.$$

The probability density per unit redshift is
$g_z=e^{-\widetilde\tau}\,d\widetilde\tau/dz$. The lecture's density per unit conformal time is $g_\tau=-\dot{\widetilde\tau}e^{-\widetilde\tau}$, so $g_\tau=H(z)g_z$ because $|dz/d\tau|=H(z)$. A change of variable changes a probability **density**, and its peak need not stay at the same redshift.

We plot both densities, retain the standard $g_\tau$ peak as the toy last-scattering epoch, and neglect reionization.

In [ ]:
z_vis = np.linspace(0.0, 2200.0, 6000)
xe_vis = saha_xe(z_vis)
n_e = xe_vis * n_H0 * (1.0 + z_vis)**3
doptical_depth_dz = c * sigma_T * n_e / ((1.0 + z_vis) * H_of_z(z_vis))
optical_depth = cumulative_trapezoid(doptical_depth_dz, z_vis, initial=0.0)
g_z = np.exp(-optical_depth) * doptical_depth_dz
# Keep the physical normalization; check it instead of forcing it to unity.
visibility_probability = np.trapz(g_z, z_vis)
g_tau = H_of_z(z_vis) * g_z
z_peak_redshift = z_vis[np.argmax(g_z)]
z_star = z_vis[np.argmax(g_tau)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
ax1.plot(z_vis, optical_depth, color='tab:purple')
ax1.axhline(1, color='0.4', ls='--')
ax1.set(xlabel='Redshift $z$', ylabel=r'Optical depth $\widetilde\tau(z)$',
        title='Thomson optical depth', xlim=(600, 1600), ylim=(0, 8))
# Normalize heights only for comparing shapes with different units.
ax2.plot(z_vis, g_z/g_z.max(), label=r'$g_z/\max(g_z)$ (per redshift)')
ax2.plot(z_vis, g_tau/g_tau.max(), ls='--',
         label=r'$g_\tau/\max(g_\tau)$ (per conformal time)')
ax2.axvline(z_star, color='0.5', ls=':', label=fr'$g_\tau$ peak: $z_*\simeq{z_star:.0f}$')
ax2.set(xlabel='Redshift $z$', ylabel='Peak-normalized shape',
        title='Saha-model visibility: two density conventions', xlim=(600, 1600))
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f'Integrated last-scattering probability: {visibility_probability:.6f}')
print(f'Per-redshift peak: z = {z_peak_redshift:.1f}')
print(f'Per-conformal-time peak (used below): z_* = {z_star:.1f}')
assert np.isclose(visibility_probability, 1.0, atol=2e-4)


Saha gives an equilibrium ionization fraction; optical depth and visibility describe photon scattering. The peak here need not reproduce the precision value near $z_*\simeq1090$, where $k_BT_*\simeq0.26$ eV. Hydrogen recombination involves departures from equilibrium, excited states, Lyman-$\alpha$ escape, and two-photon decays. The model also omits helium electrons and reionization.

The curves illustrate the rapid loss of opacity and the finite width of last scattering. Do not interpret the toy peak or its sound horizon as precision predictions. The two plotted peaks also differ slightly because their probability densities use different integration variables.

## 5. Baryon inertia, baryon loading, and the sound horizon (lecture Section 7.1)

The momentum density of a fluid is proportional to $(\rho+P)\mathbf v$. The photon inertia is $4\rho_\gamma/3$ and the baryon inertia is approximately $\rho_b$. Their ratio is

$$R(z)=\frac{3\rho_b}{4\rho_\gamma}
=\frac{3\Omega_b}{4\Omega_\gamma}\frac{1}{1+z}.$$

**Baryon inertia** is resistance to acceleration; **baryon loading** measures that inertia relative to the photons. Since $R\propto a$, baryons become more important relative to radiation even while both densities fall.

For adiabatic compression, $\delta_b=3\delta_\gamma/4$, hence $\delta\rho_b=R\delta\rho_\gamma$. Photons supply the pressure, so

$$\frac{c_s^2}{c^2}=\frac{\delta P}{\delta\rho}=\frac{1}{3(1+R)}.$$

The comoving distance travelled by sound is

$$r_s(z_*)=\int_{z_*}^\infty\frac{c_s(z)}{H(z)}\,dz.$$

We evaluate the equivalent integral over scale factor from $0$ to $a_*$, which avoids a large finite redshift cutoff. Here $z_*$ is still the **Saha-model** visibility peak.

In [ ]:
def baryon_loading(z):
    return (3.0 * Omega_b / (4.0 * Omega_gamma)) / (1.0 + z)

def sound_speed(z):
    return c / np.sqrt(3.0 * (1.0 + baryon_loading(z)))

def sound_horizon_Mpc(z_end):
    # a^2 H(a) = H0 sqrt(Omega_r + Omega_m a + Omega_Lambda a^4).
    prefactor_R = 3.0 * Omega_b / (4.0 * Omega_gamma)
    return quad(
        lambda a: c / (Mpc * H0 * np.sqrt(3.0 * (1.0 + prefactor_R*a))
                      * np.sqrt(Omega_r + Omega_m*a + Omega_Lambda*a**4)),
        0.0, 1.0/(1.0 + z_end), epsabs=1e-8, epsrel=1e-9
    )[0]

r_s_Mpc = sound_horizon_Mpc(z_star)
R_star = baryon_loading(z_star)
chi_star_Mpc = quad(lambda z: c/(Mpc*H_of_z(z)), 0, z_star)[0]
print(f'Saha-model R(z_*) = {R_star:.3f}')
print(f'Saha-model c_s(z_*)/c = {sound_speed(z_star)/c:.3f}')
print(f'Saha-model sound horizon = {r_s_Mpc:.2f} Mpc')
print(f'Saha-model chi_* = {chi_star_Mpc:.1f} Mpc')
print(f'Saha-model acoustic angle r_s/chi_* = {r_s_Mpc/chi_star_Mpc:.5f} rad')
print(f'For comparison, the same background evaluated at z=1090 gives r_s = '
      f'{sound_horizon_Mpc(1090):.2f} Mpc.')
print('Changing the endpoint is a comparison, not a non-equilibrium recombination calculation.')

z_sound = np.linspace(500, 5000, 500)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(z_sound, baryon_loading(z_sound))
ax1.set(xlabel='Redshift $z$', ylabel=r'$R=3\rho_b/(4\rho_\gamma)$',
        title='Relative baryon inertia grows as redshift falls')
ax2.plot(z_sound, sound_speed(z_sound)/c)
ax2.axhline(1/np.sqrt(3), ls='--', color='0.4', label='no baryon inertia')
ax2.set(xlabel='Redshift $z$', ylabel=r'$c_s/c$', title='Baryon loading reduces sound speed')
ax2.legend()
plt.tight_layout()
plt.show()


## 6. Perturbations and momentum exchange (lecture Sections 6–7.2)

### 6.1 Metric potentials, anisotropic stress, and Fourier modes

In conformal Newtonian gauge,

$$ds^2=a^2[-(1+2\Psi)c^2d\tau^2+(1-2\Phi)d\mathbf{x}^2].$$

The general scalar metric perturbation contains four scalar functions. Two scalar coordinate choices remove the time–space shift and scalar spatial shear, leaving $\Psi$ and $\Phi$. These are **two scalar metric potentials**, not two freely propagating gravitational degrees of freedom. Einstein's constraints relate them to matter perturbations. General relativity's two propagating gravitational polarizations belong to the tensor sector.

Anisotropic stress is directional momentum flux beyond isotropic pressure and bulk motion: $T^i{}_j=P\delta^i{}_j+\pi^i{}_j$, with $\pi^i{}_i=0$. In the scalar sector, a quadrupole in the momentum distribution sources $\Phi-\Psi$. Free-streaming neutrinos arriving from different regions develop this quadrupole. Frequent scattering suppresses it in tightly coupled photons; non-relativistic baryons and cold dark matter have small random-velocity stresses. Photons also develop anisotropic stress when tight coupling fails. A perfectly homogeneous, isotropic neutrino background has no anisotropic stress.

A Fourier mode is a sinusoidal spatial pattern, $e^{i\mathbf{k}\cdot\mathbf{x}}$, with wavelength $2\pi/k$. Spatial derivatives become $i\mathbf{k}$, and linear evolution on a homogeneous background treats each mode independently. Super-Hubble and sub-Hubble refer to $kc\ll\mathcal H$ and $kc\gg\mathcal H$; acoustic phase evolution is controlled by $kr_s$. Also $\delta_\gamma=4\Theta_0$ because $\rho_\gamma\propto T^4$.

### 6.2 Where the fluid equations come from

In the following analytic equations set $c=1$, as in the notes, and define the positive conformal scattering rate $\Gamma_c=-\dot{\widetilde\tau}$. The energy and momentum moments of the photon distribution give

$$\dot\Theta_0=-\theta_\gamma/3+\dot\Phi,$$
$$\dot\theta_\gamma=k^2(\Theta_0+\Psi)+\Gamma_c(\theta_b-\theta_\gamma),$$
$$\dot\theta_b+\mathcal H\theta_b=k^2\Psi+\frac{\Gamma_c}{R}(\theta_\gamma-\theta_b).$$

Negative velocity divergence compresses and heats the photons. The term $\dot\Phi$ describes changing spatial volume. Pressure acceleration is $-\nabla\Theta_0$, gravitational acceleration is $-\nabla\Psi$, and their divergences give the positive $k^2$ terms. The baryon term $\mathcal H\theta_b$ is expansion drag. Baryon thermal pressure and the photon quadrupole are neglected here.

Thomson scattering transfers momentum to electrons, and Coulomb interactions pass it to ions. The opposite force densities give opposite accelerations weighted by the inertias. Thus the baryon drag contains $1/R$. Adding the photon equation to $R$ times the baryon equation cancels the collision terms.

To isolate this mechanism, freeze $R$ and $\Gamma_c$ and omit all other forces for the next demonstration. With $u=\Gamma_c(\tau-\tau_i)$, the dimensionless collision-only equations are

$$\frac{d\theta_\gamma}{du}=\theta_b-\theta_\gamma,\qquad
\frac{d\theta_b}{du}=\frac{\theta_\gamma-\theta_b}{R}.$$

The conserved combination is $\theta_\gamma+R\theta_b$. The slip decays as

$$\Delta\theta(u)=\Delta\theta(0)e^{-(1+1/R)u}.$$

The numerical example uses arbitrary common units for the two velocity divergences.

In [ ]:
R_drag = 0.6
u_drag = np.linspace(0, 4, 400)
initial_velocities = [1.0, 0.0]

def collision_rhs(u, velocities):
    theta_gamma, theta_b = velocities
    slip = theta_gamma - theta_b
    return [-slip, slip/R_drag]

drag = solve_ivp(collision_rhs, (u_drag[0], u_drag[-1]), initial_velocities,
                 t_eval=u_drag, rtol=1e-10, atol=1e-12)
assert drag.success
theta_gamma_drag, theta_b_drag = drag.y
momentum_drag = theta_gamma_drag + R_drag*theta_b_drag
slip_exact = (initial_velocities[0]-initial_velocities[1]) * np.exp(-(1+1/R_drag)*u_drag)
common_velocity = (initial_velocities[0]+R_drag*initial_velocities[1])/(1+R_drag)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(u_drag, theta_gamma_drag, label=r'$\theta_\gamma$')
ax.plot(u_drag, theta_b_drag, label=r'$\theta_b$')
ax.axhline(common_velocity, color='0.4', ls='--', label='common limiting value')
ax.set(xlabel=r'$u=\Gamma_c(\tau-\tau_i)$', ylabel='Velocity divergence (common units)',
       title='Scattering locks the photon and baryon flows')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Common limiting velocity divergence: {common_velocity:.6f}')
print(f'Max change in weighted momentum: {np.max(np.abs(momentum_drag-momentum_drag[0])):.2e}')
assert np.allclose(momentum_drag, momentum_drag[0], atol=1e-10, rtol=0)
assert np.allclose(theta_gamma_drag-theta_b_drag, slip_exact, atol=1e-9, rtol=0)


## 7. Derive the oscillator and its equilibrium (lecture Sections 7.3–7.4)

### 7.1 First keep the intrinsic temperature

Add the Euler equations with weights $1,R$ **before** taking the tight-coupling limit. A small slip multiplied by a large rate can still transmit finite momentum; deleting both species' drag terms prematurely loses that transfer. With $\theta_\gamma\simeq\theta_b=\theta$,

$$(1+R)\dot\theta+\mathcal H R\theta=k^2\Theta_0+(1+R)k^2\Psi.$$

Use $\dot R=\mathcal H R$ and differentiate $\dot\Theta_0=-\theta/3+\dot\Phi$:

$$\ddot\Theta_0+\frac{\dot R}{1+R}\dot\Theta_0+k^2c_s^2\Theta_0
=\ddot\Phi+\frac{\dot R}{1+R}\dot\Phi-\frac{k^2}{3}\Psi.$$

Pressure supplies the restoring force, changing inertia supplies the first-derivative term, and gravity supplies compression and metric evolution. This is already an equation for the intrinsic photon temperature.

### 7.2 Why add the gravitational potential?

The photon leaves the plasma with intrinsic perturbation $\Theta_0$ but loses energy when climbing out of a potential well. For static potentials, the fractional gravitational contribution is $\Psi_*-\Psi_{\rm obs}$. The observer term is a direction-independent monopole. The emission contribution is therefore

$$X=\Theta_0+\Psi.$$

A well has $\Psi<0$ in our convention. $X$ combines local temperature and gravitational redshift; it is not a new degree of freedom. Doppler and integrated gravitational terms must still be added to obtain the observed anisotropy.

Substituting $\Theta_0=X-\Psi$ gives

$$\ddot X+\frac{\dot R}{1+R}\dot X+k^2c_s^2X
=\ddot\Phi+\ddot\Psi+\frac{\dot R}{1+R}(\dot\Phi+\dot\Psi)-k^2c_s^2R\Psi.$$

For frozen $R,\Phi,\Psi$, this becomes $\ddot X+k^2c_s^2(X+R\Psi)=0$. Equilibrium requires zero net pressure-plus-gravity acceleration:

$$0=\frac{k^2}{1+R}\Theta_{0,\rm eq}+k^2\Psi,\qquad
\Theta_{0,\rm eq}=-(1+R)\Psi,\qquad X_{\rm eq}=-R\Psi.$$

Photons must supply enough pressure to support both their own inertia and the baryons. In a potential well the extra effective-temperature offset is positive.

### 7.3 Solve the toy oscillator and compare with its analytic solution

Restore $c$ by using dimensionless time $s=kc(\tau-\tau_i)$ at fixed $k$. Then

$$\frac{d^2X}{ds^2}=-\frac{X+R\Psi}{3(1+R)},\qquad
X=X_{\rm eq}+(X_{\rm initial}-X_{\rm eq})
\cos\left(\frac{s}{\sqrt{3(1+R)}}\right).$$

We use a small constant well $\Psi=-10^{-5}$ and the illustrative matter-era adiabatic displacement $\Theta_{0,\rm initial}=-2\Psi/3$, with zero initial velocity. Holding that displacement fixed while varying a frozen $R$ isolates baryon loading; it is not a self-consistent evolution from the early Universe.

The first panel uses the same time variable for all curves, showing both the reduced frequency and the shifted equilibrium. The second uses acoustic phase to compare compression and rarefaction extrema.

In [ ]:
Psi_toy = -1.0e-5
Theta_initial = -2*Psi_toy/3
X_initial = Theta_initial + Psi_toy
s_osc = np.linspace(0, 30, 1200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

for R_toy in [0.0, 0.3, 0.6]:
    def oscillator_rhs(s, state):
        X, dX_ds = state
        return [dX_ds, -(X + R_toy*Psi_toy)/(3*(1+R_toy))]

    solution = solve_ivp(oscillator_rhs, (0, s_osc[-1]), [X_initial, 0.0],
                         t_eval=s_osc, rtol=1e-10, atol=1e-14)
    assert solution.success
    phase_toy = s_osc/np.sqrt(3*(1+R_toy))
    X_eq = -R_toy*Psi_toy
    X_exact = X_eq + (X_initial-X_eq)*np.cos(phase_toy)
    numerical_error = np.max(np.abs(solution.y[0]-X_exact))/abs(Psi_toy)
    assert numerical_error < 1e-7
    line, = ax1.plot(s_osc, solution.y[0]/abs(Psi_toy), label=fr'$R={R_toy:.1f}$')
    ax1.plot(s_osc[::75], X_exact[::75]/abs(Psi_toy), 'o',
             color=line.get_color(), fillstyle='none', ms=4)
    ax1.axhline(X_eq/abs(Psi_toy), color=line.get_color(), ls=':', lw=1)
    ax2.plot(phase_toy, solution.y[0]/abs(Psi_toy), color=line.get_color())
    print(f'R={R_toy:.1f}: Theta_eq/|Psi|={-(1+R_toy)*Psi_toy/abs(Psi_toy):.2f}, '
          f'X_eq/|Psi|={X_eq/abs(Psi_toy):.2f}; max numerical error/|Psi|={numerical_error:.2e}')

ax1.set(xlabel=r'$s=kc(\tau-\tau_i)$', ylabel=r'$X/|\Psi|$',
        title='Lines: numerical; circles: analytic; dotted: equilibrium')
ax1.legend()
ax2.axvline(np.pi, color='0.5', ls='--', label='first compression')
ax2.axvline(2*np.pi, color='0.5', ls=':', label='first rarefaction')
ax2.set(xlabel=r'Acoustic phase $s/\sqrt{3(1+R)}$', ylabel=r'$X/|\Psi|$',
        title='Compare extrema at the same acoustic phase', xlim=(0, 3*np.pi))
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 8. A snapshot of coherent acoustic phases

For frozen $R,\Phi,\Psi$, the solution derived above is

$$X+R\Psi=C_1\cos(kr_s)+C_2\sin(kr_s),\qquad
r_s=\int c_s\,d\tau.$$

Constant coefficients give an exact sine/cosine solution. For slowly varying coefficients, $kr_s$ approximates the accumulated phase, while the amplitude and equilibrium also evolve. Rapid gravitational driving requires the full equation.

For a displaced fluid with zero initial velocity, write $A=X_{\rm initial}+R\Psi$. Then $(X-X_{\rm eq})/A=\cos(kr_s)$ and, using $\dot\Theta_0=-\theta/3$ for constant $\Phi$,

$$\frac{\theta}{3 k c_s A}=\sin(kr_s).$$

The curves below are **normalized phase shapes**, not the absolute temperature and Doppler sources. The latter require amplitudes and projection factors. Coherent adiabatic initial conditions produce this common temporal phase despite random spatial Fourier amplitudes.

In [ ]:
k = np.linspace(0.0001, 0.16, 2000)  # Mpc^-1
phase = k * r_s_Mpc
density_source = np.cos(phase)
velocity_source = np.sin(phase)

fig, ax = plt.subplots()
ax.plot(k, density_source, lw=2, label=r'temperature displacement $(X-X_{\rm eq})/A$')
ax.plot(k, velocity_source, lw=1.7, label=r'velocity divergence $\theta/(3 k c_s A)$')
for n in range(1, 8):
    k_n = n * np.pi / r_s_Mpc
    if k_n < k.max():
        ax.axvline(k_n, color='0.7', lw=0.9, ls='--')
        ax.text(k_n, 1.08, f'n={n}', ha='center', fontsize=9)
ax.axhline(0, color='0.3', lw=0.8)
ax.set(xlabel=r'Comoving wavenumber $k\;[\mathrm{Mpc}^{-1}]$',
       ylabel='Normalized phase shape', ylim=(-1.25, 1.3),
       title='A snapshot of coherent photon–baryon oscillations')
ax.legend(loc='lower left')
plt.show()


The vertical lines satisfy $k_n r_s=n\pi$: extrema of the temperature displacement when the velocity vanishes. For the compressive adiabatic initial condition in the preceding example, odd $n$ gives compression and even $n$ rarefaction.

Projection approximately maps $k$ to $\ell\sim k\chi_*$, producing the angular acoustic pattern. Baryon loading shifts the effective-temperature equilibrium to $-R\Psi$, making the compression and rarefaction extrema unequal. Diffusion and the finite width of last scattering, omitted from the toy oscillator, suppress small-scale anisotropies.

## 9. Exercises

When changing background parameters, maintain flatness with $\Omega_\Lambda=1-\Omega_m-\Omega_r$; changing $\Omega_b$ at fixed $\Omega_c$ also changes $\Omega_m$. Rerun all downstream cells.

1. **Local conservation:** explain why radiation energy in a fixed comoving volume decreases, while vacuum energy grows. Is either in conflict with $\nabla_\mu T^{\mu\nu}=0$?
2. **Expansion and distance:** double $\Omega_r$. Predict the change in $z_{\rm eq}$ analytically. Why does an incoming photon obey $d\chi/d\tau=-c$?
3. **Saha:** change $\Omega_b$ at fixed $H_0$. Does equilibrium half ionization occur earlier or later? Explain using the density prefactor. Separately replace $E_I$ by $0.9E_I$.
4. **Momentum exchange:** vary R_drag with the same initial velocities. Predict their common final velocity from the conserved weighted momentum before rerunning.
5. **Baryon loading:** explain both effects of increasing $R$ in the oscillator: a lower frequency and a shifted equilibrium. Why is the equilibrium $-(1+R)\Psi$ for $\Theta_0$ but $-R\Psi$ for $X$?
6. **Coherence:** average $\cos^2(kr_s+\varphi)$ over many uniformly distributed temporal phases $\varphi$. Compare it with $\cos^2(kr_s)$. Random spatial amplitudes are compatible with coherent temporal phases; it is the latter phases you are randomizing here.
7. **Approximations:** identify what must change to include time-dependent potentials, photon diffusion, or non-equilibrium recombination.

Lecture 2 connects these physical sources to the angular spectrum and real Planck data.

## Summary

Local covariant conservation allows energy in a comoving volume to change as the Universe expands. Saha equilibrium predicts hydrogen formation well below the binding-energy scale, while the scattering history determines last scattering. Rapid momentum exchange locks photon and baryon velocities; photon pressure and total fluid inertia then produce acoustic oscillations. The observable emission contribution is $X=\Theta_0+\Psi$, with equilibrium $-R\Psi$ in the constant-potential toy model. Coherent phases and the sound horizon set the acoustic ruler that Lecture 2 projects onto the sky.

See [Lecture 1](Lecture_1_CMB_physics.pdf) for the corresponding derivations.